# 4 - Build Vector Store
### AI Estimation Agent - Nagaland PWD SOR 2021

Loads your `sor_clean.csv` (2101 items) and builds a ChromaDB vector database.
This allows the AI to semantically search the SOR — finding the right item
even when the wording is slightly different.

**Run every cell top to bottom. Takes about 2-5 minutes.**

## Step 1 - Install Libraries
*(Run once. Skip if already done.)*

In [1]:
!pip install chromadb sentence-transformers pandas -q
print("Libraries ready")

Libraries ready



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 - Imports

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import os
import json
print("Imports OK")

p:\AAA Project 1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Step 3 - Config
No changes needed here — just run it.

In [3]:
SOR_CSV      = "sor_clean.csv"          # your extracted SOR
VECTOR_DB    = "./sor_vectordb"         # folder where ChromaDB will be saved
COLLECTION   = "npwd_sor_2021"          # name of the collection
EMBED_MODEL  = "all-MiniLM-L6-v2"      # free, fast, ~80MB download once
CONFIG_FILE  = "vectorstore_config.json"

# Verify CSV exists
if os.path.exists(SOR_CSV):
    df = pd.read_csv(SOR_CSV)
    print(f"CSV found      : {SOR_CSV}")
    print(f"Total items    : {len(df)}")
    print(f"Columns        : {df.columns.tolist()}")
    print(f"Chapters       : {df['chapter'].nunique()}")
    print(f"Rate range     : Rs.{df['rate'].min():.2f} to Rs.{df['rate'].max():.2f}")
else:
    print(f"ERROR: {SOR_CSV} not found.")
    print("Run 3_Parse_SOR.ipynb first.")

CSV found      : sor_clean.csv
Total items    : 2101
Columns        : ['item_no', 'chapter', 'description', 'unit', 'rate']
Chapters       : 37
Rate range     : Rs.0.50 to Rs.396383.10


## Step 4 - Load Embedding Model
Downloads `all-MiniLM-L6-v2` (~80MB) on first run, then cached locally.
This model converts text into numbers (vectors) so similar descriptions match each other.

In [4]:
print(f"Loading model: {EMBED_MODEL}")
print("First run downloads ~80MB. Subsequent runs are instant.")
print()
model = SentenceTransformer(EMBED_MODEL)
print("Model loaded successfully!")

# Quick test
test = model.encode("earthwork excavation in soft soil")
print(f"Test embedding shape: {test.shape}  (384 dimensions)")

Loading model: all-MiniLM-L6-v2
First run downloads ~80MB. Subsequent runs are instant.



p:\AAA Project 1\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aloto\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11907.10it/s]


Model loaded successfully!
Test embedding shape: (384,)  (384 dimensions)


## Step 5 - Clean & Prepare Data
Standardises units and builds the search text for each SOR item.

**Search text = chapter + description + unit** — gives the AI full context.

In [5]:
df = pd.read_csv(SOR_CSV)

# Standardise units to lowercase
unit_map = {
    '100 Sqm': '100sqm', '100 Cum': '100cum', '100 Metre': '100metre',
    'Sqm': 'sqm', 'sqm': 'sqm',
    'Cum': 'cum', 'cum': 'cum',
    'Metre': 'metre', 'metre': 'metre',
    'Kg': 'kg', 'kg': 'kg',
    'Nos': 'nos', 'Each': 'each',
    'Set': 'set', 'Job': 'job',
}
df['unit_clean'] = df['unit'].map(unit_map).fillna(df['unit'].str.lower().str.strip())

# Build search text for each item
# Format: CHAPTER | description | unit
# This gives the vector search full context
df['search_text'] = (
    df['chapter'].str.strip() + ' | ' +
    df['description'].str.strip() + ' | ' +
    df['unit_clean'].str.strip()
)

# Clean item_no - ensure no duplicates
df = df.drop_duplicates(subset=['item_no'])
df = df.reset_index(drop=True)

print(f"Items ready to embed : {len(df)}")
print(f"\nSample search texts:")
for _, row in df.sample(5, random_state=42).iterrows():
    print(f"  [{row['item_no']:12s}] {row['search_text'][:85]}")

Items ready to embed : 2101

Sample search texts:
  [A26.6       ] 26.0 NEW TECHNOLOGIES AND MATERIALS | have minimum density of 1000 Kg/cum & minimum H
  [A17.31      ] 17.0 SANITARY INSTALLATIONS | ground fixed to wooden cleats with C.P. brass screws an
  [A15.55.1    ] From brick work in cement mortar | Dismantling of cement concrete platform along with
  [A17.7.6     ] 17.0 SANITARY INSTALLATIONS | with single 15 mm C.P.brass pillar tap White Vitreous C
  [A19.36.1    ] 19.0 DRAINAGE | pipes including collars/spigot jointed with stiff mixture of includin


## Step 6 - Build ChromaDB Vector Store
This is the main step. Embeds all 2101 SOR items and stores them in ChromaDB.

> Takes **2-5 minutes**. Progress shown as it runs. Do not close VS Code.

In [6]:
# Setup ChromaDB
print('Setting up ChromaDB...')
client = chromadb.PersistentClient(path=VECTOR_DB)

# Delete old collection if rebuilding fresh
try:
    client.delete_collection(COLLECTION)
    print('Deleted existing collection - rebuilding fresh')
except:
    pass

collection = client.create_collection(
    name=COLLECTION,
    metadata={'hnsw:space': 'cosine'}  # cosine similarity is best for text
)
print(f'Collection created: {COLLECTION}')
print()

# Embed and insert in batches of 100
BATCH_SIZE = 100
total = len(df)
inserted = 0

print(f'Embedding {total} SOR items in batches of {BATCH_SIZE}...')
print()

for start in range(0, total, BATCH_SIZE):
    batch = df.iloc[start : start + BATCH_SIZE]

    ids        = []
    documents  = []
    embeddings = []
    metadatas  = []

    for idx, row in batch.iterrows():
        # Make a safe unique ID
        safe_id = str(row['item_no']).strip().replace(' ', '_')
        if not safe_id:
            safe_id = f'item_{idx}'

        ids.append(safe_id)
        documents.append(str(row['search_text']))
        embeddings.append(model.encode(str(row['search_text'])).tolist())
        metadatas.append({
            'item_no':     str(row['item_no']),
            'chapter':     str(row['chapter']),
            'description': str(row['description']),
            'unit':        str(row['unit_clean']),
            'rate':        float(row['rate']),
        })

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    inserted += len(batch)
    pct = (inserted / total) * 100
    bar = '#' * int(pct // 5) + '-' * (20 - int(pct // 5))
    print(f'  [{bar}] {inserted:4d}/{total} ({pct:.0f}%)', end='\r')

print(f'\n\nDone! {inserted} items embedded and stored in ChromaDB.')

Setting up ChromaDB...
Collection created: npwd_sor_2021

Embedding 2101 SOR items in batches of 100...

  [####################] 2101/2101 (100%)

Done! 2101 items embedded and stored in ChromaDB.


## Step 7 - Verify the Vector Store
Checks the vector store was built correctly and runs 3 test searches.

In [7]:
# Check count
count = collection.count()
print(f"Items in vector store: {count}")
print(f"Expected            : {len(df)}")
print(f"Status              : {'OK' if count == len(df) else 'MISMATCH - re-run Step 6'}")
print()

Items in vector store: 2101
Expected            : 2101
Status              : OK



## Step 8 - Test Searches
This is the most important verification step.
Run searches for common work items and check if the right SOR item comes back.

In [8]:
def search_sor(query, top_k=5):
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    embedding = model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[embedding],
        n_results=top_k
    )
    for i, meta in enumerate(results['metadatas'][0]):
        dist = results['distances'][0][i]
        score = round((1 - dist) * 100, 1)  # convert distance to similarity %
        print(f"  {i+1}. [{meta['item_no']:12s}] {meta['description'][:55]:55s}")
        print(f"     Unit: {meta['unit']:8s} | Rate: Rs.{meta['rate']:>10.2f} | Match: {score}%")
    print()

# Test with typical civil engineering work items
search_sor("earthwork excavation in foundation")
search_sor("RCC M20 concrete in columns")
search_sor("brick masonry in cement mortar")
search_sor("plastering 12mm cement mortar 1:4")
search_sor("steel reinforcement bars TMT")


Query: 'earthwork excavation in foundation'
------------------------------------------------------------
  1. [A2.9.2      ] out the excavated soil and disposal of surplus excavate
     Unit: cum      | Rate: Rs.    835.80 | Match: 61.9%
  2. [A2.7.2      ] and disposal of excavated earth lead upto 50 m and lift
     Unit: cum      | Rate: Rs.    731.00 | Match: 61.3%
  3. [A2.30.3     ] each deposited layer by ramming, watering etc, disposin
     Unit: each     | Rate: Rs.    566.50 | Match: 61.1%
  4. [A2.7.1      ] and disposal of excavated earth lead upto 50 m and lift
     Unit: cum      | Rate: Rs.    398.90 | Match: 60.9%
  5. [A2.6.1      ] and disposal of excavated earth lead upto 50 m and lift
     Unit: cum      | Rate: Rs.    220.90 | Match: 60.8%


Query: 'RCC M20 concrete in columns'
------------------------------------------------------------
  1. [A5.32       ] Extra for laying RCC in or under foul positions        
     Unit: cum      | Rate: Rs.    275.60 | Match: 60

## Step 9 - Try Your Own Search
Type any work item description and see what the AI finds.

In [9]:
# Change this to any work item you want to search
MY_QUERY = "providing and fixing door frame teak wood"

search_sor(MY_QUERY, top_k=5)


Query: 'providing and fixing door frame teak wood'
------------------------------------------------------------
  1. [A14.27.1    ] including bright or/and black enamelled M.S. butt hinge
     Unit: sqm      | Rate: Rs.   2345.00 | Match: 57.9%
  2. [A14.26.2.2  ] M.S. piano hinges with necessary screws First class tea
     Unit: sqm      | Rate: Rs.   2351.90 | Match: 57.0%
  3. [A14.26.1.2  ] M.S. piano hinges with necessary screws Ist class teak 
     Unit: sqm      | Rate: Rs.   2497.50 | Match: 56.0%
  4. [A14.26.2.1  ] piano hinges with necessary screws Superior class teak 
     Unit: sqm      | Rate: Rs.   2364.90 | Match: 55.8%
  5. [A9.45       ] Providing and fixing teak wood lipping of size 25x3 mm 
     Unit: metre    | Rate: Rs.     58.20 | Match: 55.1%



## Step 10 - Save Config
Saves the vector store settings so the AI agent notebook can load it automatically.

In [ ]:
config = {
    "vector_db_path": VECTOR_DB,
    "collection_name": COLLECTION,
    "embed_model":     EMBED_MODEL,
    "total_items":     int(collection.count()),
    "sor_csv":         SOR_CSV,
}

with open(CONFIG_FILE, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config saved: {CONFIG_FILE}")
print()
print(json.dumps(config, indent=2))
print()
print("=" * 50)
print("VECTOR STORE COMPLETE!")
print("=" * 50)
print(f"  {collection.count()} SOR items indexed and searchable")
print()
print("NEXT: Open AI_agent.ipynb")
print("That is the final step - connects everything")
print("and produces the Abstract of Cost Excel.")
print("=" * 50)